In [ ]:
%pwd

In [3]:
!pip install requests tqdm
!pip install pandas pyarrow tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
import re
import glob
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
import requests
from tqdm import tqdm

url = "https://archive.org/download/coop-hourly-precipitation/v2/archive/HPD_v02r02_POR_s19400101_e20250225_c20250303.tar.gz"
filename = "HPD_hourly_precipitation.tar.gz"

response = requests.get(url, stream=True)

total_size = int(response.headers.get('content-length', 0))
block_size = 1024

with open(filename, "wb") as file:
    for data in tqdm(response.iter_content(block_size), total=total_size//block_size):
        file.write(data)

print("Download complete")

In [ ]:
import tarfile

tar = tarfile.open("HPD_hourly_precipitation.tar.gz")
tar.extractall("HPD_hourly_dataset")
tar.close()

print("Extraction complete")

In [2]:
import os
import re
import tarfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# -------------------------
# PATHS
# -------------------------
RAW_DIR = Path("HPD_hourly_dataset")
PROCESSED_DIR = Path("HPD_hourly_mm_parquet")
OUTPUT_DIR = Path("HPD_outputs")

PROCESSED_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(files)} station CSV files.")

# -------------------------
# HELPERS
# -------------------------
def clean_station_wide_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.rename(columns={
        "STATION": "station_id",
        "NAME": "station_name",
        "LATITUDE": "latitude",
        "LONGITUDE": "longitude",
        "ELEVATION": "elevation_m",
        "DATE": "date",
        "DlySum": "daily_total_raw",
        "DlySumMF": "daily_total_measurement_flag",
        "DlySumQF": "daily_total_quality_flag",
        "DlySumS1": "daily_total_source_flag_1",
        "DlySumS2": "daily_total_source_flag_2",
    })

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    hour_val_cols = [f"HR{h:02d}Val" for h in range(24)]
    for col in hour_val_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[df[col] == -9999, col] = pd.NA

    if "daily_total_raw" in df.columns:
        df["daily_total_raw"] = pd.to_numeric(df["daily_total_raw"], errors="coerce")
        df.loc[df["daily_total_raw"] == -9999, "daily_total_raw"] = pd.NA

    return df


def wide_to_hourly_precip_mm(df: pd.DataFrame) -> pd.DataFrame:
    id_cols = ["station_id", "station_name", "latitude", "longitude", "elevation_m", "date"]
    hour_val_cols = [f"HR{h:02d}Val" for h in range(24)]

    hourly = df[id_cols + hour_val_cols].melt(
        id_vars=id_cols,
        value_vars=hour_val_cols,
        var_name="hour_col",
        value_name="precip_raw"
    )

    hourly["hour"] = hourly["hour_col"].str.extract(r"HR(\d{2})Val").astype("Int64")
    hourly["datetime"] = hourly["date"] + pd.to_timedelta(hourly["hour"], unit="h")
    hourly["precip_mm"] = pd.to_numeric(hourly["precip_raw"], errors="coerce") * 0.254

    # optional sanity filter: keep missing as missing, drop impossible negatives
    hourly.loc[hourly["precip_mm"] < 0, "precip_mm"] = pd.NA

    hourly = hourly.drop(columns=["hour_col", "precip_raw"])

    hourly = hourly[
        [
            "station_id",
            "station_name",
            "latitude",
            "longitude",
            "elevation_m",
            "date",
            "hour",
            "datetime",
            "precip_mm",
        ]
    ]

    return hourly


# -------------------------
# PROCESS ALL STATIONS
# -------------------------
station_inventory = []
failed_files = []

for f in tqdm(files, desc="Processing station files"):
    try:
        raw_df = pd.read_csv(f)
        wide_df = clean_station_wide_df(raw_df)
        hourly_df = wide_to_hourly_precip_mm(wide_df)

        station_id = str(hourly_df["station_id"].iloc[0])
        station_name = str(hourly_df["station_name"].iloc[0])
        latitude = float(hourly_df["latitude"].iloc[0])
        longitude = float(hourly_df["longitude"].iloc[0])

        elev_val = hourly_df["elevation_m"].iloc[0]
        elevation_m = float(elev_val) if pd.notna(elev_val) else None

        out_file = PROCESSED_DIR / f"{station_id}.parquet"
        hourly_df.to_parquet(out_file, index=False)

        station_inventory.append({
            "station_id": station_id,
            "station_name": station_name,
            "latitude": latitude,
            "longitude": longitude,
            "elevation_m": elevation_m,
            "source_csv": f.name,
            "daily_rows": len(wide_df),
            "hourly_rows": len(hourly_df),
            "start_date": wide_df["date"].min(),
            "end_date": wide_df["date"].max(),
            "parquet_file": out_file.name,
        })

    except Exception as e:
        failed_files.append({
            "file": str(f),
            "error": str(e)
        })

inventory_df = pd.DataFrame(station_inventory)
failed_df = pd.DataFrame(failed_files)

inventory_df.to_parquet(OUTPUT_DIR / "station_inventory.parquet", index=False)
inventory_df.to_csv(OUTPUT_DIR / "station_inventory.csv", index=False)

if not failed_df.empty:
    failed_df.to_csv(OUTPUT_DIR / "failed_files.csv", index=False)

print("\nDone.")
print("Processed stations:", len(inventory_df))
print("Failed files:", len(failed_df))
print("Processed folder:", PROCESSED_DIR.resolve())
print("Inventory file:", (OUTPUT_DIR / "station_inventory.parquet").resolve())

Found 2074 station CSV files.


Processing station files:   0%|          | 0/2074 [00:00<?, ?it/s]

C:\Users\Zaman\AppData\Local\Temp\ipykernel_24608\1222040526.py:101: DtypeWarning: Columns (9,14,19,24,29,34,39,44,49,54,59,64,69,74,79,84,89,94,99,104,109,114,119,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(f)
C:\Users\Zaman\AppData\Local\Temp\ipykernel_24608\1222040526.py:101: DtypeWarning: Columns (9,14,19,24,29,34,39,44,49,54,59,64,69,74,79,84,89,94,99,104,109,114,119,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(f)
C:\Users\Zaman\AppData\Local\Temp\ipykernel_24608\1222040526.py:101: DtypeWarning: Columns (9,14,19,24,29,34,39,44,49,54,59,64,69,74,79,84,89,94,99,104,109,114,119,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(f)
C:\Users\Zaman\AppData\Local\Temp\ipykernel_24608\1222040526.py:101: DtypeWarning: Columns (9,14,19,24,29,34,39,44,49,54,59,64,69,74,79,84,89,94,99,104,109,114,119,124) have mixed types. Specif


Done.
Processed stations: 2074
Failed files: 0
Processed folder: C:\Users\Zaman\Downloads\HPD\HPD_hourly_mm_parquet
Inventory file: C:\Users\Zaman\Downloads\HPD\HPD_outputs\station_inventory.parquet


In [3]:
from pathlib import Path
import tarfile

PROCESSED_DIR = Path("HPD_hourly_mm_parquet")
OUTPUT_DIR = Path("HPD_outputs")
ARCHIVE_PATH = OUTPUT_DIR / "HPD_hourly_mm_parquet.tar.gz"

with tarfile.open(ARCHIVE_PATH, "w:gz") as tar:
    for f in PROCESSED_DIR.glob("*.parquet"):
        tar.add(f, arcname=f.name)

    # also include inventory files inside the archive
    for meta_name in ["station_inventory.parquet", "station_inventory.csv"]:
        meta_file = OUTPUT_DIR / meta_name
        if meta_file.exists():
            tar.add(meta_file, arcname=meta_file.name)

print("Created archive:")
print(ARCHIVE_PATH.resolve())
print(f"Archive size: {ARCHIVE_PATH.stat().st_size / (1024**3):.2f} GB")

Created archive:
C:\Users\Zaman\Downloads\HPD\HPD_outputs\HPD_hourly_mm_parquet.tar.gz
Archive size: 6.35 GB
